# Tutorial 8: Deployment Workflows with Kubernetes

## What You Will Learn

- Configure the Kubernetes provider in a manifest
- Understand the Dask Kubernetes Operator architecture
- Set up namespace isolation and resource quotas
- Use overlays for dev/prod Kubernetes environments
- Handle pod evictions and node failures

## Prerequisites

- Completed Tutorials 1-3
- `pip install scalable[kubernetes]`
- Kubernetes cluster access (or follow along conceptually)

> **Note:** This notebook demonstrates configuration and concepts. Actual deployment requires a running Kubernetes cluster with the Dask Operator installed.

In [ ]:
import os
import tempfile

project_dir = tempfile.mkdtemp(prefix="scalable-k8s-")
os.chdir(project_dir)
print(f"Working directory: {project_dir}")

## Step 1: Kubernetes Provider Architecture

```
ScalableSession
    └── KubernetesProvider
           └── Creates DaskCluster CR (Custom Resource)
                  └── Dask Kubernetes Operator
                         ├── Scheduler Pod
                         ├── Worker Pod (gcam-0)
                         ├── Worker Pod (gcam-1)
                         └── Worker Pod (postprocess-0)
```

The operator manages pod lifecycle, health checks, and scaling.

## Step 2: Multi-Environment Kubernetes Manifest

In [ ]:
k8s_manifest = """\
version: 1
project:
  name: climate-pipeline-k8s
  default_storage: gs://climate-artifacts/scalable-runs/

targets:
  local:
    provider: local
    max_workers: 2
    threads_per_worker: 1
    processes: false
    containers: none

  k8s-dev:
    provider: kubernetes
    namespace: climate-dev
    image: gcr.io/my-project/climate-model:latest
    adaptive:
      minimum: 1
      maximum: 5
    overlay: k8s-dev-resources

  k8s-prod:
    provider: kubernetes
    namespace: climate-prod
    image: gcr.io/my-project/climate-model:v2.1.0
    adaptive:
      minimum: 4
      maximum: 40
    overlay: k8s-prod-resources

components:
  gcam:
    image: gcr.io/my-project/gcam:7.0
    cpus: 4
    memory: 16G
    tags: [iam, climate]
    env:
      GCAM_DATA: /data/gcam

  postprocess:
    image: gcr.io/my-project/postprocess:latest
    cpus: 2
    memory: 8G
    tags: [analysis]

tasks:
  run_gcam:
    component: gcam
    cache: true
    outputs:
      database: dir

  aggregate:
    component: postprocess
    cache: true

overlays:
  k8s-dev-resources:
    components:
      gcam:
        cpus: 2
        memory: 8G
      postprocess:
        cpus: 1
        memory: 4G

  k8s-prod-resources:
    components:
      gcam:
        cpus: 16
        memory: 64G
      postprocess:
        cpus: 8
        memory: 32G
"""

with open("scalable.yaml", "w") as f:
    f.write(k8s_manifest)

print("Kubernetes manifest written with dev/prod overlays.")
print("\nTargets:")
print("  local     → development (no K8s needed)")
print("  k8s-dev   → Kubernetes dev namespace (small pods)")
print("  k8s-prod  → Kubernetes prod namespace (large pods)")

## Step 3: Validate Against Local Target

We can validate the manifest structure without K8s access by using the local target.

In [ ]:
from scalable import ScalableSession

session = ScalableSession.from_yaml("./scalable.yaml", target="local")
report = session.validate()

print(f"Manifest valid: {report.ok}")
print(f"Errors: {len(report.errors)}")
print(f"Warnings: {len(report.warnings)}")

for w in report.warnings:
    print(f"  WARN: {w.message}")

## Step 4: Check Kubernetes Provider Availability

In [ ]:
try:
    from scalable.providers.kubernetes import KubernetesProvider
    print("✓ KubernetesProvider is available")
    print(f"  Provider name: {KubernetesProvider.name}")
except ImportError:
    print("✗ KubernetesProvider not available")
    print("  Install with: pip install scalable[kubernetes]")
    print("  (requires dask-kubernetes and kubernetes packages)")

## Step 5: Kubernetes Resource Specifications

Components map directly to pod resource requests/limits in Kubernetes.

In [ ]:
from scalable.manifest.parser import load_manifest

manifest = load_manifest("./scalable.yaml")

print("Component → Pod Resource Mapping:")
print("="*50)

for name, comp in manifest.components.items():
    print(f"\n  Component: {name}")
    print(f"  Image: {comp.image or '(inherited from target)'}")
    print(f"  Resources:")
    print(f"    requests.cpu: {comp.cpus}")
    print(f"    requests.memory: {comp.memory}")
    print(f"    limits.cpu: {comp.cpus}")
    print(f"    limits.memory: {comp.memory}")
    print(f"  Labels:")
    print(f"    scalable.io/component: {name}")
    if comp.tags:
        print(f"    scalable.io/tags: {','.join(comp.tags)}")

## Step 6: Namespace and Quota Planning

In production Kubernetes, resource quotas prevent runaway usage.

In [ ]:
# Calculate total resource requirements for production
prod_overlay = manifest.raw.get("overlays", {}).get("k8s-prod-resources", {})
prod_components = prod_overlay.get("components", {})

# Merge base + overlay
max_workers = 40  # From k8s-prod target

gcam_cpus = prod_components.get("gcam", {}).get("cpus", 4)
gcam_mem = prod_components.get("gcam", {}).get("memory", "16G")
pp_cpus = prod_components.get("postprocess", {}).get("cpus", 2)
pp_mem = prod_components.get("postprocess", {}).get("memory", "8G")

# Assume 80% gcam, 20% postprocess
gcam_workers = int(max_workers * 0.8)
pp_workers = max_workers - gcam_workers

total_cpu = gcam_workers * gcam_cpus + pp_workers * pp_cpus

print("Production Resource Quota Planning:")
print(f"  Max workers: {max_workers}")
print(f"  GCAM workers: {gcam_workers} × {gcam_cpus} CPU = {gcam_workers * gcam_cpus} CPU")
print(f"  Postprocess workers: {pp_workers} × {pp_cpus} CPU = {pp_workers * pp_cpus} CPU")
print(f"  Total CPU needed: {total_cpu}")
print(f"\n  Recommended quota (with 25% headroom):")
print(f"    requests.cpu: {int(total_cpu * 1.25)}")
print(f"    pods: {max_workers + 5}  (workers + scheduler + buffer)")

In [ ]:
# Generate ResourceQuota YAML
quota_yaml = f"""\
apiVersion: v1
kind: ResourceQuota
metadata:
  name: climate-pipeline-quota
  namespace: climate-prod
spec:
  hard:
    requests.cpu: "{int(total_cpu * 1.25)}"
    requests.memory: "1280Gi"
    limits.cpu: "{int(total_cpu * 1.5)}"
    limits.memory: "1600Gi"
    pods: "{max_workers + 5}"
"""

print("Generated ResourceQuota:")
print(quota_yaml)

## Step 7: Handling Pod Evictions

Kubernetes may evict pods. Scalable catches these as `KilledWorker` exceptions.

In [ ]:
# Pattern: Retry evicted tasks
eviction_handler_code = '''
from distributed import as_completed

futures = [client.submit(run_gcam, s, tag="gcam") for s in scenarios]

results = []
evicted = []

for future in as_completed(futures):
    try:
        results.append(future.result())
    except Exception as e:
        if "KilledWorker" in type(e).__name__:
            # Pod was evicted — queue for retry
            evicted.append(future.key)
        else:
            print(f"Permanent failure: {e}")

# Retry evicted tasks (pods will be rescheduled by operator)
if evicted:
    print(f"Retrying {len(evicted)} evicted tasks...")
    retry_futures = [client.submit(run_gcam, s, tag="gcam") for s in evicted]
    retry_results = client.gather(retry_futures)
    results.extend(retry_results)
'''

print("Pod eviction handling pattern:")
print(eviction_handler_code)

## Step 8: CI/CD Integration

Automate Kubernetes deployments from GitHub Actions:

In [ ]:
ci_workflow = """\
# .github/workflows/scalable-k8s.yaml
name: Scalable K8s Pipeline
on:
  workflow_dispatch:
    inputs:
      target:
        description: "Target environment"
        default: "k8s-dev"
        type: choice
        options: [k8s-dev, k8s-prod]

jobs:
  run:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: google-github-actions/auth@v2
        with:
          credentials_json: ${{ secrets.GCP_SA_KEY }}
      - uses: google-github-actions/get-gke-credentials@v2
        with:
          cluster_name: climate-cluster
          location: us-central1
      - name: Run Pipeline
        run: |
          pip install scalable[kubernetes,cloud]
          scalable validate ./scalable.yaml
          scalable run ./scalable.yaml --target ${{ inputs.target }}
"""

print("GitHub Actions CI/CD workflow:")
print(ci_workflow)

## Step 9: Local Development with Local Target

The same manifest works locally — just select a different target.

In [ ]:
import time


def run_gcam_local(scenario: int) -> dict:
    """Simulate GCAM execution."""
    time.sleep(0.2)
    return {"scenario": scenario, "emissions": scenario * 1.5}


# Run locally — same workflow code works on K8s
session = ScalableSession.from_yaml("./scalable.yaml", target="local")
client = session.start()

futures = [client.submit(run_gcam_local, i, tag="gcam") for i in range(5)]
results = client.gather(futures)

print("Local results (same code runs on K8s):")
for r in results:
    print(f"  {r}")

session.close()

## Summary

1. Kubernetes provider creates DaskCluster CRs managed by the Dask Operator
2. Components map to pod resource requests/limits
3. Overlays customize resources for dev vs prod namespaces
4. Adaptive scaling bounds (min/max) control pod counts
5. Pod evictions are handled as retryable failures
6. Same workflow code works locally and on Kubernetes
7. CI/CD integration automates deployment

## Next Steps

- **Tutorial 9**: ML-driven pod sizing based on historical resource usage
- **Tutorial 7**: Error handling for pod evictions
- **Tutorial 10**: AI-assisted manifest migration to Kubernetes

In [ ]:
import shutil
os.chdir("/tmp")
shutil.rmtree(project_dir, ignore_errors=True)
print("Cleaned up.")